In [0]:
  %run ./secrets-template

In [0]:
%python
import os

ssh_priv_key = dbutils.secrets.get(scope="brev", key="ssh_private_key")

with open("/tmp/ssh_private_key_2", "w") as f:
    f.write(ssh_priv_key + "\n")
os.chmod("/tmp/ssh_private_key_2", 0o600)

In [0]:
%sh
ssh -i /tmp/ssh_private_key_2 -o StrictHostKeyChecking=no $SSH_USER@$BREV_IP << 'EOF'
echo == OS ==
. /etc/os-release
echo $PRETTY_NAME
echo == GPU / Driver ==
nvidia-smi
EOF

In [0]:
%sh
echo "== step 1 - system dependencies =="
# Prep the machine for ML work
ssh -i /tmp/ssh_private_key_2 -o StrictHostKeyChecking=no $SSH_USER@$BREV_IP << 'EOF'
sudo apt-get update;
sudo apt-get install -y git curl wget gnupg ca-certificates build-essential python3-dev ninja-build cmake pkg-config tmux
EOF

In [0]:
%sh
# Remove the system-installed CUDA toolkit from the Brev machine
ssh -i /tmp/ssh_private_key_2 -o StrictHostKeyChecking=no $SSH_USER@$BREV_IP << 'EOF'
sudo apt-get purge -y nvidia-cuda-toolkit || true;
sudo apt-get autoremove -y || true
EOF

In [0]:
%sh
ssh -i /tmp/ssh_private_key_2 -o StrictHostKeyChecking=no $SSH_USER@$BREV_IP << 'EOF'
sudo apt-get update;

# Download NVIDIA's GPG keyring package.
wget -q https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64/cuda-keyring_1.1-1_all.deb -O /tmp/cuda-keyring.deb;

# Verify it's actually a .deb file
file /tmp/cuda-keyring.deb;

# Installs the keyring, which also adds NVIDIA's apt repository to the system's sources list
sudo dpkg -i /tmp/cuda-keyring.deb;
sudo apt-get update;

# Install CUDA 12.4 specifically
echo == Installing CUDA toolkit 12.4 ==;
sudo apt-get install -y cuda-toolkit-12-4;
EOF

In [0]:
%sh
ssh -i /tmp/ssh_private_key_2 -o StrictHostKeyChecking=no $SSH_USER@$BREV_IP << 'EOF'
echo == PATH ==;
if ! grep -q \"cuda-12.4\" ~/.bashrc; then
  echo "Adding CUDA 12.4 to .bashrc";
  {
  echo \"\";
  echo \"# CUDA 12.4 for GR00T + flash-attn\";
  echo \"export CUDA_HOME=/usr/local/cuda-12.4\";
  echo \"export PATH=\\\$CUDA_HOME/bin:\\\$PATH\";
  echo \"export LD_LIBRARY_PATH=\\\$CUDA_HOME/lib64:\\\$LD_LIBRARY_PATH\";
  } >> ~/.bashrc;
else
  echo "CUDA 12.4 already in .bashrc, skipping";
fi;
export CUDA_HOME=/usr/local/cuda-12.4;
export PATH=/usr/local/cuda-12.4/bin:/usr/bin:/usr/sbin:/usr/local/bin:/bin:/sbin;
export LD_LIBRARY_PATH=/usr/local/cuda-12.4/lib64;
echo == nvcc should be 12.4 now ==;
which nvcc;
nvcc --version;
EOF

In [0]:
%sh
ssh -i /tmp/ssh_private_key_2 -o StrictHostKeyChecking=no $SSH_USER@$BREV_IP << 'EOF'
if ! grep -q '.local/bin' ~/.bashrc; then
echo 'export PATH=$HOME/.local/bin:$PATH' >> ~/.bashrc;
fi;
cat ~/.bashrc | grep local/bin;
EOF

In [0]:
%sh
ssh -i /tmp/ssh_private_key_2 -o StrictHostKeyChecking=no $SSH_USER@$BREV_IP << 'EOF'
export PATH="$HOME/.local/bin:/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin";
curl -LsSf https://astral.sh/uv/install.sh | /bin/sh;
uv --version;
EOF

In [0]:
%sh
ssh -i /tmp/ssh_private_key_2 -o StrictHostKeyChecking=no $SSH_USER@$BREV_IP << 'EOF'
export PATH="$HOME/.local/bin:/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin";
uv pip install wandb;
EOF